# 入门分类：从第一行代码到可复现结果

[返回 Week 2 计划](README.md) · [前期总览](../README.md) · [回看 Week 1](../week1/README.md)

这是 W2-01～W2-07 共用的一本**带做教材**。第一次按顺序读解释、预测输出、运行小段代码，再做每节的独立练习。每次学习对应一节，不要求一次跑完。

- **一份样例、一条流程：**读数据 → 划分 → 手算指标 → 训练六个候选 → 验证选择 → 冻结 → 测试 → 保存并恢复。
- **材料已放到步骤旁：**下面解释第一次实现所需概念，链接供回看和加深；无需先读完整学校课程。
- **示例运行 ≠ 个人完成：**运行全部单元格只证明样例可执行。请把独立练习与解释保存在 `work/intro-classification/classification.ipynb`；不要因为本教材跑通就勾选进度。
- **输出位置：**示例文件写到 `assignments/bootcamp/week2/outputs/tutorial/`，重跑会覆盖同名示例产物，不会写入你的个人项目。
- 使用本项目 `.venv` 的 Python 内核，需要 NumPy、pandas、scikit-learn、matplotlib、joblib；环境问题回到[环境指南](../../../docs/setup/README.md)。

数据来自 scikit-learn 内置的乳腺癌分类教学数据，不需要下载。这里的错误代价是教学假设，本项目只演示分类方法。

| 学习单元 | 本 Notebook 章节 | 首次实践产物 |
| --- | --- | --- |
| W2-01 | [定义正类、划分与 baseline](#w2-01) | 数据卡、索引、Dummy |
| W2-02 | [从四个计数写出指标](#w2-02) | 手算核对、统一评价函数 |
| W2-03 | [训练逻辑回归并保存概率](#w2-03) | 三组 C 的训练/验证记录 |
| W2-04 | [比较决策树与训练差距](#w2-04) | 三棵树、比较表、趋势图 |
| W2-05 | [只用验证集选择并冻结](#w2-05) | 阈值表、冻结配置 |
| W2-06 | [一次最终评价与错误切片](#w2-06) | 测试指标、切片、错误表 |
| W2-07 | [保存、恢复与自己讲解](#w2-07) | 模型、配置、结果与复现检查 |


## 运行准备｜找对目录，理解三个容器

`DataFrame` 是带行列名的二维表；`Series` 是带行索引的一列；NumPy 数组便于逐元素比较和计算。这里一直保留原始行号 `sample_id`，用 `.loc[索引]` 取行，避免特征与标签错位。

下面的 `Path.cwd()` 读取内核工作目录，再逐层向上寻找项目根目录。因此从项目根目录或本 Notebook 目录启动均可。Python 的 `assert` 表示“这条约定必须成立”，失败时停下来检查，不要删掉断言继续跑。


In [ ]:
from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 不写死电脑上的绝对路径；项目根目录和 Notebook 所在目录都可以启动。
start = Path.cwd().resolve()
ROOT = next(
    (p for p in [start, *start.parents]
     if (p / 'README.md').is_file() and (p / 'assignments' / 'bootcamp').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('请从本项目根目录或 assignments/bootcamp/week2/ 启动内核。')
OUT = ROOT / 'assignments' / 'bootcamp' / 'week2' / 'outputs' / 'tutorial'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print('项目根目录：', ROOT)
print('示例输出目录：', OUT)


<a id="w2-01"></a>
## W2-01｜定义正类、划分与 baseline

材料整合：[数据与实验设计](../../../course/materials/04_data_and_experiment_design.md) 解释泄漏、实体划分与时间划分；本节先把其中必要步骤实现出来。

**先建立数据卡。**一行是一个样本的细胞核测量摘要，30 个数值特征来自细针抽吸图像；目标是 malignant / benign 分类。内置表只有样本索引，没有可用的患者标识或时间信息，因此“按行分层”只是本练习的协议，不能证明患者层面或时间上相互独立。

原标签 `0=malignant`、`1=benign`。我们关注 malignant，统一重编码为 `1=malignant`（正类）、`0=benign`（负类）。正类是**你指定的关注事件**，不是“好”的意思。原表的 `target` 绝不能混入输入特征。

**先猜再跑：**`(原标签 == 0).astype(int)` 对原值 `[0, 1, 0]` 应得到什么？


In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()  # 只拿特征，dataset.frame 还包含原始 target。
X.index.name = 'sample_id'
y = (dataset.target == 0).astype(int).rename('malignant')
y.index.name = 'sample_id'
assert list(dataset.target_names) == ['malignant', 'benign']
assert X.index.equals(y.index) and 'target' not in X.columns

label_mapping = {'original_0_malignant': 1, 'original_1_benign': 0}
print('原标签名称：', list(dataset.target_names))
print('新标签：0 = benign，1 = malignant')
print('样本数、特征数：', X.shape)  # 预期 (569, 30)。
print('映射小例子：', (np.array([0, 1, 0]) == 0).astype(int))
# 数据集说明是本地字符串，可以自行展开阅读，不会发起网络请求。
print(dataset.DESCR.split('Attribute Information:')[0][-900:])


### 为什么划分两次？

训练集用于拟合参数（包括标准化均值）；验证集用于选择 C、深度与阈值；测试集用于冻结之后评价。`stratify=y` 尽量保留正负比例，但不能自动识别同一患者、重复样本或时间泄漏。

第一次留下 20% 测试行号；第二次把剩下的 80% 拿出四分之一作为验证：`80% × 25% = 20%`，余下约 60% 为训练。569 不能整除，所以人数会有取整差异。这里只保存测试成员与协议统计，W2-06 前不产生测试预测、不检查测试错误。


In [ ]:
# 先划分索引，再用相同索引取 X 和 y，保证一一对应。
trainval_ids, test_ids = train_test_split(
    X.index.to_numpy(), test_size=0.20, random_state=SEED, stratify=y,
)
train_ids, val_ids = train_test_split(
    trainval_ids, test_size=0.25, random_state=SEED, stratify=y.loc[trainval_ids],
)
split_ids = {'train': train_ids, 'validation': val_ids, 'test': test_ids}
sets = [set(ids) for ids in split_ids.values()]
assert sets[0].isdisjoint(sets[1]) and sets[0].isdisjoint(sets[2])
assert sets[1].isdisjoint(sets[2])
assert set.union(*sets) == set(X.index)

split_summary = pd.DataFrame([
    {'split': name, 'n': len(ids), 'positive_fraction': float(y.loc[ids].mean())}
    for name, ids in split_ids.items()
])
display(split_summary)  # 预期 train=341，validation=114，test=114。
X_train, y_train = X.loc[train_ids], y.loc[train_ids]
X_val, y_val = X.loc[val_ids], y.loc[val_ids]
# 这里暂不创建 X_test / y_test，帮助自己守住评价顺序。


### 审计也有边界

只在训练表检查缺失、范围、标签分布，决定是否需要预处理。本示例额外对全表做一次**精确重复行指纹检查**，用途仅是验证划分协议：同一份记录不应落入多个集合；不利用测试特征分布选模型。完全相同的行可以查到，没有相同的行不代表没有同一患者。

`duplicated()` 返回每行是否重复；`hash_pandas_object(..., index=False)` 为特征内容生成指纹，不把原行号纳入。若发现跨集合同内容记录，本示例停止，先调查身份/标签，再按实体或重复组重做划分，不能简单删除“难预测”的测试样本。


In [ ]:
train_audit = {
    'missing_cells': int(X_train.isna().sum().sum()),
    'duplicate_feature_rows': int(X_train.duplicated().sum()),
    'positive_count': int(y_train.sum()),
}
display(pd.Series(train_audit, name='train_audit'))
display(X_train.describe().loc[['min', 'max'], X_train.columns[:6]])
assert train_audit['missing_cells'] == 0, '本教材假定特征完整；有缺失需在训练 Pipeline 中加填补器。'

fingerprints = pd.util.hash_pandas_object(X, index=False)
fp_sets = {name: set(fingerprints.loc[ids]) for name, ids in split_ids.items()}
cross_split_duplicates = any(
    fp_sets[a] & fp_sets[b]
    for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]
)
assert not cross_split_duplicates, '发现跨集合重复特征记录，请先调查并修正划分协议。'

data_card = {
    'dataset': 'sklearn.datasets.load_breast_cancer',
    'unit': 'one sample of cell-nuclei measurements; patient/time IDs unavailable',
    'features': X.columns.tolist(),
    'target': 'malignant=1, benign=0',
    'label_mapping': label_mapping,
    'seed': SEED,
    'split_method': 'two stratified row splits: 20% test, then 25% of remaining for validation',
    'train_audit': train_audit,
    'cross_split_exact_feature_duplicates': bool(cross_split_duplicates),
    'scope': 'educational exercise; error costs are hypothetical',
}
(OUT / 'data_card.json').write_text(json.dumps(data_card, indent=2, ensure_ascii=False), encoding='utf-8')
(OUT / 'split_indices.json').write_text(
    json.dumps({name: ids.tolist() for name, ids in split_ids.items()}, indent=2), encoding='utf-8'
)
split_summary.to_csv(OUT / 'split_summary.csv', index=False)


### 第一条 baseline：总猜训练集多数类

`fit(X_train, y_train)` 表示只用训练数据确定模型；`predict(X_val)` 表示对没参与拟合的验证数据给标签。Dummy 的 `most_frequent` 不学习特征规律，只记训练集中最多的标签。如果复杂模型连这个起点都没超过，应先检查实现和任务定义。

先预测：本数据的负类较多，Dummy 会把所有验证样本猜成哪一类？accuracy 是正确数除以总数，它会明显高于零，但这不意味着发现了正类。


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_val_pred = dummy.predict(X_val)
dummy_val_accuracy = float(np.mean(dummy_val_pred == y_val.to_numpy()))
print('Dummy 预测过的标签：', np.unique(dummy_val_pred))
print(f'Dummy 验证 accuracy：{dummy_val_accuracy:.3f}')
assert np.unique(dummy_val_pred).size == 1


### 独立练习与完成证据

在个人 Notebook 写下：一行代表什么；`原0→新1` 为什么成立；FP 与 FN 各是什么错误；为何两次划分得到约 60/20/20。自己重写一次索引互斥和覆盖检查，保留数据卡与 split 摘要。

<details><summary>卡住再看：标签与划分解释</summary>

FP 是把 benign 报成 malignant；FN 是把 malignant 漏报成 benign。两次划分后的训练占比为 `(1−0.2)×(1−0.25)=0.6`。索引不重叠仅说明没有复用同一行，不能证明没有复用同一患者。

</details>


In [ ]:
# TODO（个人练习，不影响示例顺序运行）：
# 在自己的 Notebook 用一句话分别说明 train、validation、test 的职责。
# 再用三个 assert 重写互斥检查；不要只复制显示出来的人数。


<a id="w2-02"></a>
## W2-02｜从四个计数写出指标

材料整合：[分类与指标 §1～2](../../../course/materials/06_classification_and_metrics.md)。以下先用手工小例子，让指标不是黑箱函数。

| 真实标签 / 预测标签 | 预测 0 | 预测 1 |
| --- | --- | --- |
| 真实 0（benign） | TN：正确判负 | FP：误报 |
| 真实 1（malignant） | FN：漏报 | TP：正确判正 |

`accuracy=(TP+TN)/总数`；`precision=TP/(TP+FP)` 问“报正的里面有多少真是正”；`recall=TP/(TP+FN)` 问“真实正类找回多少”；`F1=2TP/(2TP+FP+FN)` 同时考虑 precision 和 recall。

**先手算：**`TN=50, FP=10, FN=5, TP=35` 时四个指标是多少？对于没有正预测的模型，precision 分母为 0，数学上未定义。本教材把数值记作 0 以便列表，同时保存 `precision_defined=False`；不要把约定的 0 当成可估计的正预测质量。recall、F1 也保留同类标记。


In [ ]:
def metrics_from_counts(tn, fp, fn, tp):
    """将四个计数转成指标；值为 0 时仍用 *_defined 标记是否可定义。"""
    n = tn + fp + fn + tp
    def safe_ratio(numerator, denominator):
        return float(numerator / denominator) if denominator else 0.0
    return {
        'n': int(n), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'accuracy': safe_ratio(tn + tp, n), 'accuracy_defined': bool(n),
        'precision': safe_ratio(tp, tp + fp), 'precision_defined': bool(tp + fp),
        'recall': safe_ratio(tp, tp + fn), 'recall_defined': bool(tp + fn),
        'F1': safe_ratio(2 * tp, 2 * tp + fp + fn),
        'F1_defined': bool(2 * tp + fp + fn),
        # 教学假设：一次漏报的成本是一次误报的 5 倍，不是临床结论。
        'cost': int(5 * fn + fp),
    }

hand_example = metrics_from_counts(tn=50, fp=10, fn=5, tp=35)
display(pd.Series(hand_example))
assert np.isclose(hand_example['accuracy'], 0.85)
assert np.isclose(hand_example['precision'], 35 / 45)
assert np.isclose(hand_example['recall'], 35 / 40)
assert np.isclose(hand_example['F1'], 70 / 85)


### 用布尔数组数出四格

`truth == 1` 会得到 True/False；`&` 对两个同长度数组逐项取“并且”；布尔数组 `.sum()` 就是真值的个数。注意这里用 `&`，不是用于单个真假值的 Python `and`。每个比较都加括号。

以后所有模型共用下面的函数，避免一张表里用不同的指标规则。给同一份标签和预测，手算函数应与 scikit-learn 的结果一致。


In [ ]:
def classification_metrics(y_true, y_pred):
    truth = np.asarray(y_true, dtype=int)
    pred = np.asarray(y_pred, dtype=int)
    assert truth.shape == pred.shape and truth.ndim == 1
    assert np.isin(truth, [0, 1]).all() and np.isin(pred, [0, 1]).all()
    tn = int(((truth == 0) & (pred == 0)).sum())
    fp = int(((truth == 0) & (pred == 1)).sum())
    fn = int(((truth == 1) & (pred == 0)).sum())
    tp = int(((truth == 1) & (pred == 1)).sum())
    return metrics_from_counts(tn, fp, fn, tp)

# 一个每格恰好出现一次的小样本。
toy_truth = np.array([0, 0, 1, 1])
toy_pred = np.array([0, 1, 0, 1])
assert [classification_metrics(toy_truth, toy_pred)[k] for k in ['TN', 'FP', 'FN', 'TP']] == [1, 1, 1, 1]
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
assert np.array_equal(confusion_matrix(toy_truth, toy_pred, labels=[0, 1]), [[1, 1], [1, 1]])
manual = classification_metrics(y_val, dummy_val_pred)
assert np.isclose(manual['precision'], precision_score(y_val, dummy_val_pred, zero_division=0))
assert np.isclose(manual['recall'], recall_score(y_val, dummy_val_pred, zero_division=0))
assert np.isclose(manual['F1'], f1_score(y_val, dummy_val_pred, zero_division=0))
display(pd.DataFrame([{'model': 'Dummy', **manual}]))


In [ ]:
# 正类只有 5 个时，总猜负类的 accuracy 很高，但 recall 为 0。
imbalanced_truth = np.array([0] * 95 + [1] * 5)
all_negative = np.zeros(100, dtype=int)
imbalance_result = classification_metrics(imbalanced_truth, all_negative)
display(pd.Series(imbalance_result))
assert imbalance_result['accuracy'] == 0.95 and imbalance_result['recall'] == 0.0
assert imbalance_result['precision_defined'] is False


### 在看候选分数之前写死选择规则

主标准是 `cost = 5×FN + FP`，越低越好；同分先少漏报，再按事先确定的简单模型偏好顺序比较，接着选离 0.5 更近的阈值，最后选更小阈值打破完全平局。

本练习的模型偏好顺序固定为 `LR C=0.1 → LR C=1 → LR C=10 → Tree depth=2 → 4 → 8`。**跨模型家族不存在通用的复杂度排序**，这里约定线性模型优先；同族内 C 较小正则更强、树较浅通常更简单。这个约定在看到验证分数前确定，之后不改。Dummy 只作参考，候选集合固定为这六个配置。


In [ ]:
CANDIDATES = [
    {'model': 'lr_C0.1', 'family': 'logistic', 'C': 0.1},
    {'model': 'lr_C1', 'family': 'logistic', 'C': 1.0},
    {'model': 'lr_C10', 'family': 'logistic', 'C': 10.0},
    {'model': 'tree_d2', 'family': 'tree', 'max_depth': 2},
    {'model': 'tree_d4', 'family': 'tree', 'max_depth': 4},
    {'model': 'tree_d8', 'family': 'tree', 'max_depth': 8},
]
MODEL_PREFERENCE = {spec['model']: rank for rank, spec in enumerate(CANDIDATES)}
THRESHOLDS = np.round(np.arange(0.1, 1.0, 0.1), 1)
SELECTION_ORDER = ['cost', 'FN', 'model_preference', 'distance_to_0.5', 'threshold']
selection_protocol = {
    'main_metric': 'cost = 5 * FN + FP; lower is better',
    'tie_breaks_in_order': SELECTION_ORDER,
    'model_preference': MODEL_PREFERENCE,
    'thresholds': THRESHOLDS.tolist(),
    'candidates': CANDIDATES,
    'dummy_role': 'reference only, not a tuned candidate',
    'threshold_rule': 'predict 1 when positive_probability >= threshold',
    'zero_division': 'store numerical 0 and corresponding *_defined=False',
}
(OUT / 'selection_protocol.json').write_text(
    json.dumps(selection_protocol, indent=2, ensure_ascii=False), encoding='utf-8'
)


### 独立练习与完成证据

换成 `TN=70, FP=10, FN=15, TP=5`，不看函数先手算四个指标和 cost，再用代码核对。解释为何提高 accuracy 可能同时增加教学代价。保存手算步骤，不只保存计算机输出。

<details><summary>核对答案</summary>

accuracy=0.75，precision=1/3，recall=0.25，F1=2/7，cost=85。把一个漏报改为正确判正可减少 5 成本；把一个误报改为正确判负只减少 1 成本，accuracy 却都只提高一个样本的比例。

</details>


In [ ]:
# TODO：自己从 TN=70, FP=10, FN=15, TP=5 推出指标，然后再调用函数核对。
# TODO：构造一个没有真实正类的切片，说明 recall_defined 为什么应该为 False。


<a id="w2-03"></a>
## W2-03｜训练逻辑回归并保存概率

材料整合：[线性模型 §3、§5](../../../course/materials/05_linear_models.md)；[Week 1 流程与划分](../week1/day5/05_ML_workflow_and_splits.ipynb)。

逻辑回归首先算分数 `z=w·x+b`，再通过 `sigmoid(z)=1/(1+exp(-z))` 输出 0 到 1 之间的数，作为正类概率估计；最后由阈值决定标签。模型学习系数与截距，阈值是训练后另选的决策规则。概率估计没有自动保证校准：`0.8` 不代表当前数据里一定有严格的 80% 正类频率。

**先猜再跑：**分数为负、零、正时，sigmoid 分别在 0.5 的哪边？阈值 0.5 降到 0.3 后，概率 0.4 的样本会怎样？


In [ ]:
toy_scores = np.array([-2.0, 0.0, 2.0])
toy_probabilities = 1 / (1 + np.exp(-toy_scores))
display(pd.DataFrame({
    'score': toy_scores, 'p_positive': toy_probabilities,
    'label_at_0.5': (toy_probabilities >= 0.5).astype(int),
}))
assert np.isclose(toy_probabilities[1], 0.5)


### Pipeline 把训练数据边界放进代码

不同特征的单位与范围不同。`StandardScaler` 在训练数据上计算每列均值与标准差，执行 `(x−均值)/标准差`。验证数据只能用这些训练统计量转换。先在全表 fit scaler、再划分会泄漏。

`Pipeline` 将“标准化 → 分类器”串起来；对 Pipeline 调用 `fit` 时两步都只看到传入的训练表。`C` 是正则强度的倒数：较小 C 更强地限制系数；其余设置固定，才知道改变来自 C。

优化器需要逐步找参数。出现 `ConvergenceWarning` 表示本次没有可靠完成收敛，本教材把它升级成异常并停下来，不把未收敛结果悄悄放入候选表。若触发，先检查尺度，再在**训练/验证阶段**统一增加所有逻辑回归的 `max_iter`，重跑比较，并把新设置写入记录。


In [ ]:
models = {}          # 名称 -> 已在训练集拟合的模型。
validation_probs = {}  # 名称 -> 带验证 sample_id 的正类概率，W2-05 直接复用。
comparison_rows = []

# classes_ 决定 predict_proba 的列顺序，不直接假定第二列就是本项目正类。
def positive_probability(model, features):
    classes = np.asarray(model.classes_)
    positive_columns = np.flatnonzero(classes == 1)
    assert len(positive_columns) == 1, '模型必须认识编码为 1 的正类。'
    probabilities = model.predict_proba(features)[:, int(positive_columns[0])]
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    return probabilities

for spec in CANDIDATES[:3]:
    model = Pipeline([
        ('scale', StandardScaler()),
        ('classifier', LogisticRegression(C=spec['C'], solver='lbfgs', max_iter=3000,
                                           tol=1e-4, random_state=SEED)),
    ])
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        model.fit(X_train, y_train)  # 没有传入验证或测试数据。
    models[spec['model']] = model
    val_p = positive_probability(model, X_val)
    validation_probs[spec['model']] = pd.Series(val_p, index=X_val.index, name=spec['model'])
    for split_name, features, truth in [('train', X_train, y_train), ('validation', X_val, y_val)]:
        probabilities = positive_probability(model, features)
        metrics = classification_metrics(truth, (probabilities >= 0.5).astype(int))
        comparison_rows.append({'model': spec['model'], 'split': split_name,
                                'threshold': 0.5, 'C': spec['C'], **metrics})

display(pd.DataFrame(comparison_rows)[['model', 'split', 'accuracy', 'recall', 'F1', 'cost']])


In [ ]:
example_lr = models['lr_C1']
scaler = example_lr.named_steps['scale']
# 验证训练统计量来源：样本数等于训练人数；均值等于训练表均值。
assert int(scaler.n_samples_seen_) == len(X_train)
assert np.allclose(scaler.mean_, X_train.mean(axis=0).to_numpy())
print('classes_：', example_lr.classes_)
print('scaler 拟合样本数：', scaler.n_samples_seen_)

five_ids = X_val.index[:5]
five_probs = validation_probs['lr_C1'].loc[five_ids]
display(pd.DataFrame({
    'true_label': y_val.loc[five_ids],
    'p_malignant': five_probs,
    'label_at_0.5': (five_probs >= 0.5).astype(int),
}))
# 对 Pipeline 内部的原始 score 做 sigmoid，与 predict_proba 核对。
scores = example_lr.decision_function(X_val.loc[five_ids])
assert np.allclose(1 / (1 + np.exp(-scores)), five_probs.to_numpy())


### 独立练习与完成证据

选一个验证样本，画出“特征 → 训练均值标准化 → 线性 score → sigmoid → 概率 → 阈值 → 标签”；用代码核对最后两步。写出 `C=0.1` 与 `C=10` 哪个正则更强，并引用 `n_samples_seen_` 和 `mean_` 的检查证明 scaler 只见过训练数据。

<details><summary>卡住再看：为什么不能用验证集均值？</summary>

验证集代表尚未参与学习的数据。在验证表上重新 fit scaler 会让验证阶段使用另一套坐标，也使拟合步骤接触评估数据。正确调用是用已经 fit 的 Pipeline `predict_proba(X_val)`，内部只 transform，不再 fit。

</details>


In [ ]:
# TODO：在个人实验本选择一个 five_ids 之外的验证 sample_id，展示完整流程。
# TODO：写一句“模型参数、C、阈值”分别在哪个阶段确定；不要读取测试预测。


<a id="w2-04"></a>
## W2-04｜比较决策树与训练差距

材料整合：[树与集成 §2](../../../course/materials/07_trees_and_ensembles.md)。本节只实现单棵决策树，森林与 boosting 留到后面。

决策树反复问“某特征是否小于某阈值”，把样本分到叶子，再用叶子里的训练标签比例输出概率。分裂取决于排序和阈值，不依赖欧氏距离，所以通常无需标准化。最大深度越大，模型可以写出更细的分支，也更容易记住训练噪声。

训练表现改善而验证表现变差，是过拟合的一个信号；本次随机划分未出现这个趋势也要照实记录。树的重要性只说明模型用了什么，不证明某特征造成标签变化。

**预期性质：**同一划分下浅树通常叶子较少。先观察实际结果，不要求每个验证指标都随深度单调变化。


In [ ]:
for spec in CANDIDATES[3:]:
    model = DecisionTreeClassifier(max_depth=spec['max_depth'], random_state=SEED)
    model.fit(X_train, y_train)
    models[spec['model']] = model
    val_p = positive_probability(model, X_val)
    validation_probs[spec['model']] = pd.Series(val_p, index=X_val.index, name=spec['model'])
    for split_name, features, truth in [('train', X_train, y_train), ('validation', X_val, y_val)]:
        probabilities = positive_probability(model, features)
        metrics = classification_metrics(truth, (probabilities >= 0.5).astype(int))
        comparison_rows.append({
            'model': spec['model'], 'split': split_name, 'threshold': 0.5,
            'max_depth': spec['max_depth'], 'leaf_count': model.get_n_leaves(), **metrics,
        })

# Dummy 与六个候选使用同一 split 和同一指标函数。
for split_name, features, truth in [('train', X_train, y_train), ('validation', X_val, y_val)]:
    comparison_rows.append({'model': 'Dummy', 'split': split_name, 'threshold': 0.5,
                            **classification_metrics(truth, dummy.predict(features))})
comparison = pd.DataFrame(comparison_rows)
display(comparison[['model', 'split', 'leaf_count', 'accuracy', 'recall', 'F1', 'cost']])
comparison.to_csv(OUT / 'model_comparison_at_0.5.csv', index=False)


In [ ]:
tree_rows = comparison[comparison['model'].str.startswith('tree_')]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for split_name in ['train', 'validation']:
    rows = tree_rows[tree_rows['split'] == split_name].sort_values('max_depth')
    axes[0].plot(rows['max_depth'], 1 - rows['accuracy'], marker='o', label=split_name)
    # train 与 validation 人数不同，比较代价时用平均每个样本的成本。
    axes[1].plot(rows['max_depth'], rows['cost'] / rows['n'], marker='o', label=split_name)
for ax, ylabel in zip(axes, ['Error rate', 'Cost per sample']):
    ax.set(xlabel='Maximum depth', ylabel=ylabel, xticks=[2, 4, 8])
    ax.legend()
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / 'tree_depth_comparison.png', dpi=140)
plt.show()

# 概率缓存保存原验证索引和标签，后续阈值搜索不需要重 fit 六次模型。
validation_cache = pd.DataFrame(validation_probs)
validation_cache.insert(0, 'true_label', y_val)
assert validation_cache.index.equals(X_val.index) and not validation_cache.isna().any().any()
validation_cache.to_csv(OUT / 'validation_probabilities.csv', index_label='sample_id')


### 独立练习与完成证据

引用表中两行数字解释：深度增加后训练和验证分别如何变化。若观察到训练代价下降、验证代价上升，指出哪个深度区间；若没有，直接写没有。再解释为什么图中跨 split 比较采用 `cost/n`，而同一验证集的候选排序可以直接用总 cost。

<details><summary>卡住再看：总 cost 为什么不适合跨集合直接比较？</summary>

即使每个样本的错误倾向相同，训练集人数约为验证集的三倍，总 cost 也可能更大。平均成本或错误率消除了人数的直接影响。本练习候选都在同一验证集比较，人数一致，使用总 cost 排序没有这个问题。

</details>


In [ ]:
# TODO：写一段引用具体数值的观察，不使用“树越深一定越差”的预设结论。
# TODO：找一棵树读取 get_depth() / get_n_leaves()，解释它们与 max_depth 的区别。


<a id="w2-05"></a>
## W2-05｜只用验证集选择并冻结

材料整合：[分类与指标 §4](../../../course/materials/06_classification_and_metrics.md)。

固定概率后，阈值下降会把更多样本判正，FN 不会增加，FP 不会减少；但 precision 不保证逐步单调变化。模型拟合不变，改变的是动作规则。

下面遍历六组已缓存验证概率与九个预定阈值，一共 54 行。没有新训练、没有新特征、没有测试结果。反复选择会让最佳验证分数显得乐观，所以还要保留最终测试。


In [ ]:
toy_p = np.array([0.2, 0.4, 0.7])
display(pd.DataFrame({'p': toy_p, 'at_0.5': (toy_p >= 0.5).astype(int),
                      'at_0.3': (toy_p >= 0.3).astype(int)}))
# 只有概率落在 [0.3, 0.5) 的样本会从负变正。
assert np.array_equal((toy_p >= 0.3) != (toy_p >= 0.5), [False, True, False])


In [ ]:
threshold_rows = []
for spec in CANDIDATES:
    name = spec['model']
    probabilities = validation_probs[name]
    assert probabilities.index.equals(y_val.index)
    for threshold in THRESHOLDS:
        pred = (probabilities.to_numpy() >= threshold).astype(int)
        threshold_rows.append({
            'model': name, 'family': spec['family'],
            'configuration': json.dumps({k: v for k, v in spec.items() if k not in ['model', 'family']}),
            'threshold': float(threshold),
            'model_preference': MODEL_PREFERENCE[name],
            # round 避免浮点误差让 0.4 与 0.6 的距离看起来不一样。
            'distance_to_0.5': round(abs(float(threshold) - 0.5), 8),
            **classification_metrics(y_val, pred),
        })
threshold_table = pd.DataFrame(threshold_rows)
assert len(threshold_table) == 54
ranked = threshold_table.sort_values(SELECTION_ORDER, kind='stable').reset_index(drop=True)
selected = ranked.iloc[0]
selected_name = str(selected['model'])
selected_threshold = float(selected['threshold'])
selected_model = models[selected_name]
display(ranked[['model', 'configuration', 'threshold', 'TN', 'FP', 'FN', 'TP',
                'precision', 'recall', 'F1', 'cost']].head(10))
ranked.to_csv(OUT / 'threshold_search_validation.csv', index=False)


In [ ]:
# 同一个训练好的模型，仅比较阈值改变带来的 FP / FN / cost 变化。
threshold_change = []
for label, threshold in [('default', 0.5), ('selected', selected_threshold)]:
    pred = (validation_probs[selected_name].to_numpy() >= threshold).astype(int)
    threshold_change.append({'rule': label, 'threshold': threshold,
                             **classification_metrics(y_val, pred)})
display(pd.DataFrame(threshold_change)[['rule', 'threshold', 'FP', 'FN', 'recall', 'precision', 'cost']])
print('选中的模型：', selected_name)
print('选中的阈值：', selected_threshold)
print('注意：这里是选择后的验证结果，还不是最终测试表现。')


### 测试前冻结模型、阈值与切片

“冻结”表示现在写清楚全部决定，下一节只执行，不再根据测试错误改 C、深度、特征或阈值。这里继续用在训练集上拟合的原模型；若临时把训练与验证合并重拟合，概率可能改变，旧阈值不再对应之前选过的模型。

错误分析切片也先规定：用**训练集** `mean radius` 中位数，将测试样本分成 `<中位数` 和 `≥中位数` 两个互补组。这只是教学分析分组，不是病理界限，更不是用测试错误寻找的最佳切点。


In [ ]:
radius_cutoff = float(X_train['mean radius'].median())
selected_spec = next(spec for spec in CANDIDATES if spec['model'] == selected_name)
frozen_config = {
    'status': 'frozen_before_test_evaluation',
    'selected_model': selected_name,
    'candidate_spec': selected_spec,
    'threshold': selected_threshold,
    'features': X_train.columns.tolist(),
    'positive_label': 1,
    'label_mapping': label_mapping,
    'fit_data': 'train only; no refit after selection',
    'seed': SEED,
    'preprocessing': 'StandardScaler fitted on train' if selected_spec['family'] == 'logistic' else 'none',
    'estimator_params': selected_model.named_steps['classifier'].get_params()
        if selected_spec['family'] == 'logistic' else selected_model.get_params(),
    'selection_protocol': selection_protocol,
    'selected_validation_metrics': classification_metrics(
        y_val, (validation_probs[selected_name].to_numpy() >= selected_threshold).astype(int)),
    'slice': {'feature': 'mean radius', 'cutoff': radius_cutoff, 'cutoff_source': 'training median',
              'groups': ['below_train_median', 'at_or_above_train_median']},
    'dummy_final_evaluation': 'already fitted most_frequent baseline; no selection',
}
frozen_text = json.dumps(frozen_config, indent=2, ensure_ascii=False)
(OUT / 'frozen_config.json').write_text(frozen_text, encoding='utf-8')
print('已写入冻结记录：', OUT / 'frozen_config.json')
print('训练 mean radius 中位数：', radius_cutoff)


### 独立练习与完成证据

先在纸上画出概率 0.2、0.4、0.7 的阈值变化，再解释选中模型的代价变化由 FP 还是 FN 驱动。写清楚全部平局规则，保存自己的配置表；阈值最优点为 0.5 也属于正常结果。

<details><summary>自检：哪些行为会破坏当前实验？</summary>

看测试 FN 太多后降低阈值、看测试错误后删特征、看测试切片后更改切点，都会把测试集变成下一轮验证集。可以把发现记作未来假设；若要继续调参，需要新的独立测试数据。

</details>


In [ ]:
# TODO：在个人项目画出阈值变化前后的 FP / FN，并解释代价变化。
# TODO：用文字说明“同一模型改变阈值”和“改 C 后重新 fit”为什么不同。


<a id="w2-06"></a>
## W2-06｜一次最终评价与错误切片

材料整合：[分类与指标 §6](../../../course/materials/06_classification_and_metrics.md)。

现在才打开测试预测。代码先读取磁盘上的冻结配置，确认它与内存里的选择一致，再对冻结模型和 Dummy 各预测一次。之后的指标、切片与错误分析都复用同一份预测。

“只评价一次”是实验规则：可以为了复现同一个冻结方案重跑整本教材，但不能把测试分数反馈到选择步骤。为了读懂指标而重复显示同一缓存表，不等于再调一轮模型。


In [ ]:
saved_config = json.loads((OUT / 'frozen_config.json').read_text(encoding='utf-8'))
assert saved_config['selected_model'] == selected_name
assert saved_config['threshold'] == selected_threshold
assert saved_config['features'] == X_train.columns.tolist()
assert saved_config['slice']['cutoff'] == radius_cutoff

X_test, y_test = X.loc[test_ids], y.loc[test_ids]
test_probabilities = positive_probability(selected_model, X_test)
test_predictions = (test_probabilities >= saved_config['threshold']).astype(int)
dummy_test_predictions = dummy.predict(X_test)
selected_test_metrics = classification_metrics(y_test, test_predictions)
dummy_test_metrics = classification_metrics(y_test, dummy_test_predictions)
final_metrics = pd.DataFrame([
    {'model': selected_name, 'threshold': selected_threshold, **selected_test_metrics},
    {'model': 'Dummy', 'threshold': None, **dummy_test_metrics},
])
display(final_metrics[['model', 'threshold', 'n', 'TN', 'FP', 'FN', 'TP',
                       'accuracy', 'precision', 'recall', 'F1', 'cost']])
print('混淆矩阵行是真实 0/1，列是预测 0/1：')
display(pd.DataFrame(
    [[selected_test_metrics['TN'], selected_test_metrics['FP']],
     [selected_test_metrics['FN'], selected_test_metrics['TP']]],
    index=['true_0', 'true_1'], columns=['pred_0', 'pred_1'],
))


In [ ]:
# 建一张行级结果表，后续都从这张缓存表计算，不再调用 fit 或选阈值。
test_results = pd.DataFrame({
    'true_label': y_test,
    'p_malignant': test_probabilities,
    'predicted_label': test_predictions,
    'dummy_predicted_label': dummy_test_predictions,
    'mean radius': X_test['mean radius'],
})
test_results['error'] = test_results['true_label'] != test_results['predicted_label']
test_results['error_type'] = np.select(
    [(test_results['true_label'] == 1) & test_results['error'],
     (test_results['true_label'] == 0) & test_results['error']],
    ['FN', 'FP'], default='correct',
)
test_results['slice'] = np.where(test_results['mean radius'] < radius_cutoff,
                                  'below_train_median', 'at_or_above_train_median')
slice_rows = []
for group in saved_config['slice']['groups']:
    rows = test_results[test_results['slice'] == group]
    metrics = classification_metrics(rows['true_label'], rows['predicted_label'])
    slice_rows.append({'slice': group, **metrics,
                       'error_rate': float(rows['error'].mean()) if len(rows) else None})
slice_report = pd.DataFrame(slice_rows)
assert slice_report['n'].sum() == len(X_test)
assert slice_report['FN'].sum() == selected_test_metrics['FN']
assert slice_report['FP'].sum() == selected_test_metrics['FP']
display(slice_report[['slice', 'n', 'FP', 'FN', 'error_rate', 'recall', 'recall_defined', 'cost']])


### 错误样本该怎么讲？

先陈述事实：真实标签、概率、阈值、最终预测，以及属于哪个预定切片。再提出可验证的假设，例如“这个样本的概率接近阈值，轻微模型变化可能改变标签”。不能仅凭一个错误就断言标注错了、某特征导致疾病，或为了结果好看删除样本。

为了避免挑故事，下面按 `sample_id` 排序展示前三个错误；总错误不足三个就全部显示。切片很小时，错误率的差异只能提供下一步调查线索。


In [ ]:
errors = test_results[test_results['error']].sort_index().copy()
errors['distance_to_threshold'] = abs(errors['p_malignant'] - selected_threshold)
# 这只是描述性提示，不声称证明了错误原因。
errors['inspection_note'] = np.where(
    errors['distance_to_threshold'] < 0.10,
    '概率接近冻结阈值；可作为边界样本调查，原因尚未验证',
    '概率距离冻结阈值较远；需调查数据与模型限制，原因尚未验证',
)
print(f'测试共有 {len(errors)} 个错误，下面展示 {min(3, len(errors))} 个。')
display(errors.head(3)[['true_label', 'p_malignant', 'predicted_label', 'error_type',
                       'mean radius', 'slice', 'distance_to_threshold', 'inspection_note']])
if len(errors) == 0:
    print('本次没有错误样本；不制造错误，也不因此推断未来无错误。')

test_results.to_csv(OUT / 'test_predictions.csv', index_label='sample_id')
slice_report.to_csv(OUT / 'test_slices.csv', index=False)
errors.to_csv(OUT / 'test_errors.csv', index_label='sample_id')
metrics_artifact = {
    'evaluation_split': 'held_out_test', 'selected_model': selected_name,
    'threshold': selected_threshold, 'selected': selected_test_metrics,
    'dummy': dummy_test_metrics, 'slice_cutoff_from_train': radius_cutoff,
    'zero_division_rule': selection_protocol['zero_division'],
}
(OUT / 'metrics.json').write_text(json.dumps(metrics_artifact, indent=2), encoding='utf-8')


### 独立练习与完成证据

用自己的话解释至少一个 FP 或 FN；区分“表中直接看见的事实”和“需要更多证据的原因假设”。报告两组切片的样本数、错误率，说明小样本限制。个人项目保存测试指标、错误案例与冻结记录。

<details><summary>一种合格的解释结构</summary>

“样本 i 的真实标签是 1，冻结阈值为 t，模型概率 p 小于 t，因此产生 FN，增加 5 个教学成本。它位于事先定义的半径切片 A。概率接近阈值可能提示边界不稳定，但不能据此断言测量或标签错误。本轮不再改阈值。”用你自己的实际数字替换 i、t、p、A。

</details>


In [ ]:
# TODO：个人实验本中挑选按 sample_id 排序的前三个错误（不足三个就全部），
# 每个写“事实 / 假设 / 怎样进一步验证”，不要利用它们重新调参。


<a id="w2-07"></a>
## W2-07｜保存、恢复与自己讲解

材料整合：本 Notebook 的数据卡、选择规则、冻结配置与评价表，不额外增加阅读。

模型文件保存的是训练过的对象；JSON 配置解释这个对象为什么被选中，以及阈值是什么。只保存模型文件会丢失阈值和正类约定；只保存分数则无法重做预测。因此把模型、阈值、特征顺序、标签映射一起保存，并保留可读 JSON 和 CSV。

joblib 加载会恢复 Python 对象，这里只加载本单元刚刚生成的本地文件。恢复后对**验证集**概率逐项核对，无需再次使用测试结果来证明文件可读。


In [ ]:
environment = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
    'joblib': joblib.__version__,
}
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
model_bundle = {
    'model': selected_model, 'threshold': selected_threshold,
    'feature_names': X_train.columns.tolist(),
    'positive_label': 1, 'label_mapping': label_mapping,
    'frozen_config': frozen_config,
}
joblib.dump(model_bundle, OUT / 'classification_bundle.joblib')
restored = joblib.load(OUT / 'classification_bundle.joblib')
restored_val_p = positive_probability(restored['model'], X_val[restored['feature_names']])
original_val_p = validation_probs[selected_name].to_numpy()
np.testing.assert_allclose(restored_val_p, original_val_p, rtol=0, atol=1e-12)
assert restored['threshold'] == selected_threshold
assert np.array_equal((restored_val_p >= restored['threshold']).astype(int),
                      (original_val_p >= selected_threshold).astype(int))
print('恢复检查通过：验证概率与阈值分类均一致。')
display(pd.Series(environment, name='environment'))


In [ ]:
# 生成的是教学样例报告骨架，不代替你对结果和局限的个人解释。
report_lines = [
    '# 分类带做示例运行记录', '',
    '这是教学代码生成的运行记录；不代表个人练习已完成。', '',
    f'- 数据：load_breast_cancer；特征数 {X.shape[1]}；malignant=1。',
    f'- 划分：seed={SEED}；train/validation/test={len(train_ids)}/{len(val_ids)}/{len(test_ids)}。',
    f'- 冻结模型：{selected_name}；阈值：{selected_threshold}。',
    '- 选择：只用验证集；cost → FN → 预定模型偏好 → 距离 0.5 → 较小阈值。',
    f"- 测试：FP={selected_test_metrics['FP']}，FN={selected_test_metrics['FN']}，cost={selected_test_metrics['cost']}。",
    f"- Dummy 测试成本：{dummy_test_metrics['cost']}。", '',
    '## 复现', '',
    '在项目 .venv 内核从头运行 classification_walkthrough.ipynb；同名示例产物会覆盖。',
    '保留 split_indices.json、selection_protocol.json、frozen_config.json 与 environment.json。', '',
    '## 限制', '',
    '单次随机按行划分；缺少患者/时间标识；教学假设成本；验证选择有乐观偏差；小切片不能支持因果结论。',
    '下一步发现只写成假设，本轮不依据测试结果更改模型。', '',
    '## 个人讲解（请在自己的项目补充）', '',
    '1. 为什么选择这组模型与阈值？引用验证 FP/FN/cost。',
    '2. 三个错误样本有哪些事实，哪些只是原因假设？',
    '3. 训练与验证趋势怎样？有没有观察到过拟合？',
]
(OUT / 'RUN_REPORT.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
display(pd.DataFrame([
    {'file': p.name, 'bytes': p.stat().st_size}
    for p in sorted(OUT.iterdir()) if p.is_file()
]))


### 独立完成：把带做变成自己的项目

1. 在 `work/intro-classification/classification.ipynb` 保留同样七节，自己写数据卡、手算过程、选择理由和错误解释。
2. 关闭参考答案，用另一组四格计数算 precision、recall、F1 和 cost，再核对。
3. 从干净内核运行自己的项目；检查索引、scaler 训练样本数、缓存行号、冻结配置、模型恢复一致性。
4. 用 10 分钟讲清：训练/验证/测试的职责；C 与阈值的区别；为什么 accuracy 不够；这个模型为何被选中；测试发现为什么不能直接反馈调参。
5. 按[通用项目评分](../../../course/ASSESSMENT.md#project-rubric)自评。独立成果达到要求，再去[进度表](../../../course/TRACKER.md)记录 0b；运行本样例本身不算通过。

**最常见的卡点：**索引错位回 W2-01；正类与 FP/FN 混乱回 W2-02；scaler fit 范围回 W2-03；训练好验证差回 W2-04；阈值与 C 混淆回 W2-05；只报平均分、不看样本量回 W2-06。

完成后进入 [Week 3：从梯度到小网络](../week3/README.md)。
